# Work IQ actions

The [Work IQ MCP notebook](workiq-mcp.ipynb) covers read tools. This optional notebook focuses on the four tools that change Microsoft 365 data: `create_entity`, `update_entity`, `delete_entity`, and `do_action`.

**Running the action cells changes real Microsoft 365 data without further prompts.** Using the same fixtures seeded into the lab mailbox, you create a Contoso Seattle restock-review event, update it, delete only that notebook-created event, and send a follow-up email to the signed-in lab user. The event has no attendees, and no mail is sent to the fictional fixture authors.

> In the Skillable lab, Work IQ authentication is already configured.

## Step 1: Authenticate as the signed-in user

Load the app registration from `.env`. Sign in with your lab account in the browser when prompted.

In [ ]:
import json
import os

import httpx
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from msal import PublicClientApplication

load_dotenv(override=True)

app_id = os.environ["WORK_IQ_SEARCH_ENTRA_APP_ID"]
tenant_id = os.environ["WORK_IQ_SEARCH_ENTRA_TENANT_ID"]

app = PublicClientApplication(
    client_id=app_id,
    authority=f"https://login.microsoftonline.com/{tenant_id}",
)
auth_result = app.acquire_token_interactive(["api://workiq.svc.cloud.microsoft/WorkIQAgent.Ask"])
if "access_token" not in auth_result:
    raise RuntimeError(f"Authentication failed: {auth_result.get('error_description', auth_result)}")

token = auth_result["access_token"]
print("Signed in to Work IQ")


async def call_workiq(tool_name, arguments):
    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token}"},
        timeout=httpx.Timeout(30, read=180),
    ) as http_client:
        async with streamable_http_client(
            "https://workiq.svc.cloud.microsoft/mcp", http_client=http_client
        ) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(tool_name, arguments)
    if result.isError:
        raise RuntimeError(f"Work IQ {tool_name} failed: {result.content}")
    return result


def event_from_result(result):
    payloads = [result.structuredContent]
    for content in result.content:
        if content.type == "text":
            try:
                payloads.append(json.loads(content.text))
            except json.JSONDecodeError:
                continue
    while payloads:
        payload = payloads.pop()
        if isinstance(payload, str):
            try:
                payloads.append(json.loads(payload))
            except json.JSONDecodeError:
                continue
        elif isinstance(payload, dict):
            if isinstance(payload.get("id"), str) and payload["id"].strip():
                return payload
            payloads.extend(payload.get(key) for key in ("body", "data", "value"))
    return None

## Step 2: Inspect write tools and schemas

List the write tools and their input schemas, then inspect the create schema for calendar events. The collection path `/me/events` supports create; update targets an individual event instead. These discovery calls do not change data.

- `create_entity` takes a collection's `parentUrl`.
- `update_entity` and `delete_entity` take an item's `entityUrl`.
- `do_action` takes an `actionUrl`, such as `/me/sendMail`.
- `jsonBody` is a JSON-encoded string, so encode payloads with `json.dumps`.

In [ ]:
async with httpx.AsyncClient(
    headers={"Authorization": f"Bearer {token}"},
    timeout=httpx.Timeout(30, read=180),
) as http_client:
    async with streamable_http_client("https://workiq.svc.cloud.microsoft/mcp", http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_response = await session.list_tools()
            for tool in tools_response.tools:
                if tool.name in {"create_entity", "update_entity", "delete_entity", "do_action"}:
                    print(tool.name, json.dumps(tool.inputSchema, indent=2))

            for operation in ("create",):
                result = await session.call_tool("get_schema", {"path": "/me/events", "operationType": operation})
                if result.isError:
                    raise RuntimeError(f"Work IQ get_schema failed: {result.content}")
                print(f"Calendar event {operation} schema:")
                for content in result.content:
                    if content.type == "text":
                        print(content.text)

## Step 3: Create a Seattle restock-review event

The tenant seeding script sends the three files in `data/workiq-fixtures` to the lab user's mailbox. Read those same fixtures to build a follow-up for the Professional Claw Hammer 16oz stockout (SKU `HTHM001600`). Running this cell creates a 30-minute review tomorrow at 15:00 UTC, with no attendees, and captures its ID automatically. Open the event in your calendar before continuing.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

if globals().get("created_event_result") is not None:
    raise ValueError("A practice event was already created. Continue to update or cleanup instead of creating a duplicate.")

fixture_dir = Path.cwd() / "data" / "workiq-fixtures"
if not fixture_dir.is_dir():
    fixture_dir = Path.cwd().parent / "data" / "workiq-fixtures"
fixture_names = (
    "urgent-professional-claw-hammer-out-of-stock.txt",
    "weekly-inventory-report-seattle-flagged.txt",
    "customer-escalation-hammer-unavailable.txt",
)
seeded_context = "\n\n---\n\n".join(
    (fixture_dir / name).read_text(encoding="utf-8").strip() for name in fixture_names
)
lab_email = auth_result.get("id_token_claims", {}).get("preferred_username", "").strip()
if "@" not in lab_email:
    raise ValueError("The signed-in user's UPN is missing. Rerun authentication with your lab account.")

review_subject = "Contoso Seattle: Professional Claw Hammer restock review"
review_agenda = (
    "Proposed next actions for Professional Claw Hammer 16oz (HTHM001600):\n"
    "- Verify Seattle stock and exact availability at Bellevue, Redmond, Tacoma, and Online.\n"
    "- Assess a store-to-store transfer for the contractor's five-unit request.\n"
    "- Escalate to procurement if available stock cannot cover the request.\n"
    "- Confirm a fulfillment plan with customer support; no transfer is approved by this notebook."
)
start = (datetime.now(timezone.utc) + timedelta(days=1)).replace(hour=15, minute=0, second=0, microsecond=0)
end = start + timedelta(minutes=30)
create_args = {
    "parentUrl": "/me/events",
    "jsonBody": json.dumps({
        "subject": review_subject,
        "body": {"contentType": "Text", "content": f"{review_agenda}\n\nSeeded emails:\n{seeded_context}"},
        "start": {"dateTime": start.strftime("%Y-%m-%dT%H:%M:%S"), "timeZone": "UTC"},
        "end": {"dateTime": end.strftime("%Y-%m-%dT%H:%M:%S"), "timeZone": "UTC"},
    }),
}
print(json.dumps(create_args, indent=2))

created_event_result = await call_workiq("create_entity", create_args)
created_event = event_from_result(created_event_result)
event_id = created_event["id"] if created_event else ""
if not event_id:
    print(created_event_result.model_dump_json(indent=2))
    raise RuntimeError("The event was created, but its ID is missing from the response. Do not create it again.")
print(f"Created: {review_subject} | {start:%Y-%m-%d %H:%M} UTC | 30 minutes | no attendees")
print(f"Event ID: {event_id}")
if created_event.get("webLink"):
    print(f"Open event: {created_event['webLink']}")

## Step 4: Add transfer coordination to the review

Using only the event ID captured in step 3, extend the review to 45 minutes and update its subject to include transfer coordination. This changes the calendar event, not inventory or transfer orders. Refresh your calendar before cleanup.

In [ ]:
if globals().get("created_event_result") is None or not globals().get("event_id", "").strip():
    raise ValueError("Create the practice event in step 3 and capture its event_id before updating it.")

updated_subject = "Contoso Seattle: HTHM001600 restock and transfer coordination"
updated_end = start + timedelta(minutes=45)
update_args = {
    "entityUrl": f"/me/events/{event_id}",
    "jsonBody": json.dumps({
        "subject": updated_subject,
        "end": {"dateTime": updated_end.strftime("%Y-%m-%dT%H:%M:%S"), "timeZone": "UTC"},
    }),
}
print(json.dumps(update_args, indent=2))

update_result = await call_workiq("update_entity", update_args)
print(f"Updated: {updated_subject} | {start:%Y-%m-%d %H:%M} UTC | 45 minutes")
print(f"Event ID: {event_id}")

## Step 5: Clean up the practice event

Running this cell deletes only the event created in step 3, using its captured ID. It does not modify the seeded emails or any pre-existing calendar events. Successful cleanup clears the ID so rerunning update or delete cannot accidentally reuse it.

In [ ]:
if globals().get("created_event_result") is None or not globals().get("event_id", "").strip():
    raise ValueError("There is no captured practice event to delete. Create one in step 3 first.")

delete_args = {"entityUrl": f"/me/events/{event_id}"}
print(json.dumps(delete_args, indent=2))

delete_result = await call_workiq("delete_entity", delete_args)
print(f"Deleted notebook-created Seattle restock event: {event_id}")
event_id = ""
created_event_result = None

## Step 6: Send the Seattle restock follow-up

`do_action` invokes an action such as sending mail, replying, or accepting an event. This example uses the [Microsoft Graph sendMail request](https://learn.microsoft.com/graph/api/user-sendmail).

Send a follow-up based on the seeded stockout emails to the signed-in lab user's UPN, obtained automatically during authentication. The fictional senders are context only, not recipients. The message describes proposed actions, not a completed transfer or verified stock counts. Check your inbox and Sent Items after running this cell. A successful send is not repeated in the same kernel.

In [ ]:
if globals().get("send_result") is not None:
    raise ValueError("The Seattle restock follow-up was already sent in this kernel. Check your inbox and Sent Items.")

recipient = lab_email
send_args = {
    "actionUrl": "/me/sendMail",
    "jsonBody": json.dumps({
        "message": {
            "subject": "Contoso Seattle: HTHM001600 restock follow-up",
            "body": {
                "contentType": "Text",
                "content": f"Lab follow-up based on the seeded Microsoft 365 emails.\n\n{review_agenda}\n\nSources:\n{seeded_context}",
            },
            "toRecipients": [{"emailAddress": {"address": recipient}}],
        },
        "saveToSentItems": True,
    }),
}
print(json.dumps(send_args, indent=2))

send_result = await call_workiq("do_action", send_args)
print(f"Send request accepted for {recipient}: Contoso Seattle restock follow-up. Check your inbox and Sent Items.")

## Recap

You used the seeded Contoso Seattle stockout emails to create a restock-review event, extend it for transfer coordination, delete that notebook-created event, and send a follow-up to your own lab mailbox. Running each action cell performs its write without an input prompt. No inventory was changed, no transfer was approved, and no fictional colleagues received invitations or mail. If you skipped cleanup, return to step 5 to remove your event.

**Optional next step:** [Work IQ A2A](workiq-a2a.ipynb) demonstrates delegating a task to Work IQ as a remote agent.